# Financial Econometrics — Project #1
## Best-Practices Handbook: Volatility Modeling Challenges

> **Context:** This handbook is written for quants, traders, and risk managers on a derivatives desk. It addresses four common problems that arise when modeling volatility from financial time series data. Each challenge is examined through definition, demonstration, diagnosis, damage assessment, and remediation.

---

### Challenges Covered
1. Overfitting
2. Skewness
3. Sensitivity to Outliers
4. Lack of Interpretation

---

**Dataset:** 5 years of daily S&P 500 adjusted close prices (`^GSPC`) via `yfinance`. Log returns are computed as the base dataset throughout.

In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────────
# Install required packages (run once; comment out if already installed)
!pip install yfinance arch scipy statsmodels --quiet

In [ ]:
# ── Global Imports ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import yfinance as yf
import scipy.stats as stats
from scipy.stats import skew, kurtosis, jarque_bera, t as t_dist

import statsmodels.api as sm
from statsmodels.stats.stattools import jarque_bera as sm_jb
from statsmodels.graphics.tsaplots import plot_acf

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline

from arch import arch_model

# ── Plot Style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = ['#1f4e79', '#e06c00', '#2e7d32', '#c62828', '#6a1b9a']

print('All packages imported successfully.')

In [ ]:
# ── Data Download ─────────────────────────────────────────────────────────────
# Pull 5 years of S&P 500 daily data
ticker = '^GSPC'
raw = yf.download(ticker, start='2019-01-01', end='2024-01-01', progress=False)

# Compute log returns: r_t = ln(P_t / P_{t-1})
prices = raw['Close'].squeeze()          # ensure 1-D Series
log_returns = np.log(prices / prices.shift(1)).dropna()
log_returns.name = 'Log Return'

print(f'Ticker  : {ticker}')
print(f'Period  : {log_returns.index[0].date()} → {log_returns.index[-1].date()}')
print(f'N obs   : {len(log_returns):,}')
print(f'Mean    : {log_returns.mean():.6f}')
print(f'Std Dev : {log_returns.std():.6f}')
print(f'Skewness: {skew(log_returns):.4f}')
print(f'Kurtosis: {kurtosis(log_returns):.4f}  (excess)') 

---

# Challenge 1 — Overfitting

---

## 1.1 Definition

A model is said to **overfit** when it minimises in-sample loss $\mathcal{L}_{\text{train}}$ at the expense of elevated out-of-sample loss $\mathcal{L}_{\text{test}}$.

For a polynomial regression of degree $d$ fitted to returns $\{r_t\}$:

$$\hat{r}_t = \beta_0 + \beta_1 t + \beta_2 t^2 + \cdots + \beta_d t^d + \varepsilon_t$$

Overfitting occurs when $d$ is chosen so large that $\hat{\sigma}^2_{\text{train}} \ll \hat{\sigma}^2_{\text{test}}$, i.e.:

$$\text{MSE}_{\text{train}} \ll \text{MSE}_{\text{test}}$$

In the GARCH context, overfitting manifests when the model order $(p, q)$ is inflated beyond what the data can support, leading to unstable parameter estimates across rolling windows.

## 1.2 Description

Overfitting happens when a model learns the noise in historical data rather than the true underlying signal, making it perform well in-sample but fail to generalize to new observations. In volatility modeling, an over-parameterized model will fit past return fluctuations precisely but produce wildly inaccurate variance forecasts for future dates.

In [ ]:
# ── 1.3 Demonstration: Overfitting in Polynomial Regression on Log Returns ────
# We use a subset of log returns as the 'signal' and fit polynomial models
# of increasing degree to illustrate the bias-variance tradeoff.

# Use 200 observations for a clear illustration
sub = log_returns.iloc[:200].reset_index(drop=True)
X = np.arange(len(sub)).reshape(-1, 1)   # time index as feature
y = sub.values

# Train / test split (chronological — no shuffling for time series)
split = 150
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

degrees = [1, 3, 8, 15]
results = []

for d in degrees:
    pipe = make_pipeline(PolynomialFeatures(d), LinearRegression())
    pipe.fit(X_train, y_train)
    mse_tr = mean_squared_error(y_train, pipe.predict(X_train))
    mse_te = mean_squared_error(y_test,  pipe.predict(X_test))
    results.append({'Degree': d, 'MSE Train': mse_tr, 'MSE Test': mse_te})

df_ov = pd.DataFrame(results)
df_ov['Generalization Gap'] = df_ov['MSE Test'] - df_ov['MSE Train']
print(df_ov.to_string(index=False))

In [ ]:
# ── 1.4 Diagram: Overfitting Visualized ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=False)
fig.suptitle('Challenge 1 — Overfitting: Polynomial Fit to S&P 500 Log Returns',
             fontsize=14, fontweight='bold', y=1.01)

X_full = np.arange(len(sub)).reshape(-1, 1)

for ax, d in zip(axes.flat, degrees):
    pipe = make_pipeline(PolynomialFeatures(d), LinearRegression())
    pipe.fit(X_train, y_train)

    ax.scatter(X_train, y_train, s=4, alpha=0.5, color=PALETTE[0], label='Train')
    ax.scatter(X_test,  y_test,  s=4, alpha=0.5, color=PALETTE[1], label='Test')
    ax.plot(X_full, pipe.predict(X_full), color=PALETTE[3], lw=1.5,
            label=f'Poly deg {d}')
    ax.axvline(split, color='grey', ls='--', lw=0.8, label='Train/Test split')

    mse_tr = mean_squared_error(y_train, pipe.predict(X_train))
    mse_te = mean_squared_error(y_test,  pipe.predict(X_test))
    ax.set_title(f'Degree {d} | Train MSE: {mse_tr:.2e} | Test MSE: {mse_te:.2e}')
    ax.set_xlabel('Trading Day Index')
    ax.set_ylabel('Log Return')
    ax.legend(loc='upper right', fontsize=7)

plt.tight_layout()
plt.savefig('overfitting_diagram.png', bbox_inches='tight')
plt.show()
print('Figure saved: overfitting_diagram.png')

## 1.5 Diagnosis

| Diagnostic Tool | What to Look For |
|---|---|
| **Train vs. Test MSE** | Large gap between in-sample and out-of-sample error |
| **Learning Curves** | Training error decreasing while validation error rises |
| **Cross-Validation** | High variance across folds |
| **AIC / BIC** | Prefer lower-order models that minimize information criteria |
| **Parameter Stability** | Coefficients that swing wildly across rolling estimation windows |

In [ ]:
# ── Diagnosis: Time-Series Cross-Validation MSE by Degree ─────────────────────
tscv = TimeSeriesSplit(n_splits=5)
cv_results = {}

for d in [1, 3, 6, 10, 15]:
    pipe = make_pipeline(PolynomialFeatures(d), LinearRegression())
    scores = -cross_val_score(pipe, X, y, cv=tscv,
                              scoring='neg_mean_squared_error')
    cv_results[d] = scores

fig, ax = plt.subplots(figsize=(8, 4))
means = [np.mean(cv_results[d]) for d in cv_results]
stds  = [np.std(cv_results[d])  for d in cv_results]
degs  = list(cv_results.keys())

ax.errorbar(degs, means, yerr=stds, fmt='-o', color=PALETTE[0],
            ecolor=PALETTE[1], elinewidth=2, capsize=5, lw=2,
            label='CV MSE ± 1 std')
ax.set_title('Diagnosis: CV MSE vs. Polynomial Degree (Time-Series CV)')
ax.set_xlabel('Polynomial Degree')
ax.set_ylabel('Mean Squared Error')
ax.legend()
plt.tight_layout()
plt.show()
print('Optimal degree is near the minimum of the CV MSE curve above.')

## 1.6 Damage

An overfit volatility model produces variance forecasts that are excessively reactive to historical idiosyncrasies rather than representative of future market conditions. In practice this means:

- **Mispriced options**: If implied volatility is anchored to an overfit realized volatility estimate, options will be systematically overpriced or underpriced, exposing the desk to adverse selection.
- **Flawed delta/vega hedges**: Greeks computed from an inflated or artificially smooth volatility surface will lead to under- or over-hedging, crystallizing P&L losses when the market moves.
- **VaR underestimation**: Risk systems relying on an overfit model may understate tail risk precisely when it matters most — during market stress events.

## 1.7 Directions

| Remedy | How It Helps |
|---|---|
| **Regularization (Ridge / Lasso)** | Shrinks large coefficients, penalizing model complexity |
| **Time-Series Cross-Validation** | Selects model order based on genuine out-of-sample performance |
| **Information Criteria (AIC, BIC)** | Parsimony-based order selection for GARCH/ARMA models |
| **Walk-Forward Validation** | Simulates live deployment; detects decay in forecast accuracy over time |
| **Parsimonious GARCH(1,1)** | Empirically, GARCH(1,1) is nearly optimal for equity return series — higher orders rarely improve out-of-sample performance |

---

# Challenge 2 — Skewness

---

## 2.1 Definition

**Skewness** is the third standardized central moment of a distribution. For a return series $\{r_t\}_{t=1}^T$:

$$\text{Skew}(r) = \frac{\mu_3}{\sigma^3} = \frac{\frac{1}{T}\sum_{t=1}^{T}(r_t - \bar{r})^3}{\left(\frac{1}{T}\sum_{t=1}^{T}(r_t - \bar{r})^2\right)^{3/2}}$$

- **Symmetric distribution**: $\text{Skew} = 0$ (e.g., Normal)
- **Negative (left) skew**: $\text{Skew} < 0$ — large negative returns are more common than symmetric returns (typical for equity indices)
- **Positive (right) skew**: $\text{Skew} > 0$

## 2.2 Description

Skewness captures the asymmetry of the return distribution around its mean. Equity index returns exhibit persistent negative skewness — large drawdowns occur more frequently than a symmetric Normal distribution would predict — which directly affects the pricing of puts versus calls.

In [ ]:
# ── 2.3 Demonstration: Skewness of S&P 500 Log Returns ───────────────────────
empirical_skew = skew(log_returns)
empirical_kurt = kurtosis(log_returns)  # excess kurtosis

# Jarque-Bera test for normality (H0: Normal)
jb_stat, jb_p = jarque_bera(log_returns)

print('=== Skewness Diagnostics for S&P 500 Log Returns ===')
print(f'Sample Skewness  : {empirical_skew:>10.4f}  (Normal = 0)')
print(f'Excess Kurtosis  : {empirical_kurt:>10.4f}  (Normal = 0)')
print(f'Jarque-Bera Stat : {jb_stat:>10.2f}')
print(f'Jarque-Bera p-val: {jb_p:>10.6f}  (H0: Normal)')
print()
if jb_p < 0.05:
    print('► Reject normality. Returns are significantly non-normal.')
else:
    print('► Cannot reject normality.')

# Percentile comparison: actual vs Normal
mu, sigma = log_returns.mean(), log_returns.std()
for pct in [1, 5, 95, 99]:
    actual = np.percentile(log_returns, pct)
    normal = stats.norm.ppf(pct/100, mu, sigma)
    print(f'  {pct:>2}th pct — Actual: {actual:+.4f} | Normal: {normal:+.4f} | Diff: {actual-normal:+.4f}')

In [ ]:
# ── 2.4 Diagram: Skewness Visualization ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Challenge 2 — Skewness in S&P 500 Daily Log Returns',
             fontsize=14, fontweight='bold')

# --- Panel A: Histogram vs. Normal ---
ax = axes[0]
x_grid = np.linspace(log_returns.min(), log_returns.max(), 300)
normal_pdf = stats.norm.pdf(x_grid, mu, sigma)

ax.hist(log_returns, bins=80, density=True, color=PALETTE[0],
        alpha=0.6, label='Empirical')
ax.plot(x_grid, normal_pdf, color=PALETTE[3], lw=2, label='Normal fit')
ax.axvline(log_returns.mean(), color='grey', ls='--', lw=1, label='Mean')
ax.axvline(log_returns.median(), color=PALETTE[2], ls='--', lw=1, label='Median')
ax.set_title(f'Return Distribution\nSkew = {empirical_skew:.3f}')
ax.set_xlabel('Log Return')
ax.set_ylabel('Density')
ax.legend()

# --- Panel B: QQ Plot ---
ax = axes[1]
(osm, osr), (slope, intercept, r) = stats.probplot(log_returns, dist='norm')
ax.scatter(osm, osr, s=3, alpha=0.4, color=PALETTE[0])
ax.plot(osm, slope*np.array(osm)+intercept, color=PALETTE[3], lw=2,
        label='Normal reference')
ax.set_title('Normal Q-Q Plot\n(Deviation = non-normality)')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.legend()

# --- Panel C: Rolling Skewness ---
ax = axes[2]
roll_skew = log_returns.rolling(63).skew()   # ~quarterly window
ax.plot(roll_skew.index, roll_skew, color=PALETTE[0], lw=1, label='63-day rolling skew')
ax.axhline(0, color='grey', ls='--', lw=1, label='Symmetric = 0')
ax.axhline(empirical_skew, color=PALETTE[3], ls='--', lw=1, label=f'Full-sample skew ({empirical_skew:.2f})')
ax.set_title('Rolling Skewness (63-Day Window)')
ax.set_xlabel('Date')
ax.set_ylabel('Skewness')
ax.legend()

plt.tight_layout()
plt.savefig('skewness_diagram.png', bbox_inches='tight')
plt.show()

## 2.5 Diagnosis

| Diagnostic | Interpretation |
|---|---|
| **Sample skewness coefficient** | Values outside $[-0.5, 0.5]$ indicate meaningful asymmetry |
| **Jarque-Bera test** | Tests joint normality (skewness + kurtosis); small p-value ⇒ non-normal |
| **Q-Q plot** | Systematic departure from the 45° line, especially in tails |
| **Rolling skewness** | Persistent negative rolling skewness confirms structural asymmetry |
| **Histogram vs. Normal overlay** | Visual gap between empirical left tail and fitted Normal |

## 2.6 Damage

Assuming symmetric (Normal) returns when the true distribution is negatively skewed leads to:

- **Underpriced put options**: Black-Scholes assumes log-Normal returns (zero skew). Negative skewness means downside tail risk is systematically underpriced, creating losses when selling puts.
- **Biased Value-at-Risk**: A Gaussian VaR at the 99th percentile will underestimate realized losses during market dislocations.
- **Incorrect hedge ratios**: Delta and higher-order Greeks calibrated under a symmetric distribution will be systematically wrong for portfolios with asymmetric exposures (e.g., structured products).

## 2.7 Directions

| Remedy | Mechanism |
|---|---|
| **Skewed-Normal / Skewed-t distribution** | Directly parameterizes asymmetry in GARCH innovation distributions |
| **GJR-GARCH (TGARCH)** | Captures asymmetric volatility response (larger impact from negative shocks) |
| **Johnson SU distribution** | Flexible 4-parameter family; accommodates arbitrary skewness and kurtosis |
| **Cornish-Fisher VaR expansion** | Adjusts Normal quantiles for observed skewness and kurtosis |
| **Stochastic Volatility with leverage** | Models negative correlation between return and volatility innovations |

---

# Challenge 3 — Sensitivity to Outliers

---

## 3.1 Definition

An **outlier** in a return series is an observation $r_t$ satisfying:

$$|r_t - \bar{r}| > k \cdot \hat{\sigma}, \quad k \in \{3, 4, 5\}$$

**Sensitivity to outliers** in OLS regression arises because the estimator minimises the sum of *squared* residuals:

$$\hat{\boldsymbol{\beta}}_{\text{OLS}} = \arg\min_{\boldsymbol{\beta}} \sum_{t=1}^T (r_t - \mathbf{x}_t^\top \boldsymbol{\beta})^2$$

The quadratic penalty gives outliers disproportionate influence proportional to $(r_t - \hat{r}_t)^2$, distorting coefficient estimates. The **influence function** of OLS is unbounded:

$$\text{IF}(r_t; \hat{\boldsymbol{\beta}}) \propto (r_t - \hat{r}_t) \quad \text{(unbounded as } r_t \to \pm\infty\text{)}$$

## 3.2 Description

Because OLS penalizes errors quadratically, even a single extreme return (such as a flash crash or a circuit-breaker day) can dramatically shift estimated regression coefficients and volatility parameters. Robust estimation methods bound the influence of any individual observation, preventing tail events from corrupting the entire model.

In [ ]:
# ── 3.3 Demonstration: Outlier Impact on Regression Slope ────────────────────
# Build a simple predictive regression: r_t ~ r_{t-1} (AR(1) on returns)
r_lag = log_returns.shift(1).dropna()
r_cur = log_returns[r_lag.index]

X_ar = r_lag.values.reshape(-1, 1)
y_ar = r_cur.values

# Identify extreme outliers (|return| > 4 sigma)
thresh = 4 * log_returns.std()
outlier_mask = np.abs(y_ar) > thresh
n_outliers = outlier_mask.sum()

print(f'Outlier threshold (4σ): ±{thresh:.4f}')
print(f'Number of outliers    : {n_outliers}')
print(f'Outlier dates         :')
for d in r_cur[outlier_mask].index:
    print(f'  {d.date()}  return = {r_cur[d]:+.4f}')

# OLS with and without outliers
ols_full = LinearRegression().fit(X_ar, y_ar)
ols_clean = LinearRegression().fit(X_ar[~outlier_mask], y_ar[~outlier_mask])

print(f'\nOLS slope (all data)   : {ols_full.coef_[0]:.6f}')
print(f'OLS slope (no outliers): {ols_clean.coef_[0]:.6f}')
print(f'Slope change due to outliers: {ols_full.coef_[0] - ols_clean.coef_[0]:+.6f}')

In [ ]:
# ── 3.4 Diagram: Outlier Sensitivity ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Challenge 3 — Sensitivity to Outliers: S&P 500 AR(1) Regression',
             fontsize=14, fontweight='bold')

# --- Panel A: Scatter with regression lines ---
ax = axes[0]
ax.scatter(X_ar[~outlier_mask], y_ar[~outlier_mask],
           s=4, alpha=0.3, color=PALETTE[0], label='Normal obs')
ax.scatter(X_ar[outlier_mask], y_ar[outlier_mask],
           s=60, color=PALETTE[3], marker='*', zorder=5, label='Outliers (|r|>4σ)')

x_line = np.linspace(X_ar.min(), X_ar.max(), 100).reshape(-1, 1)
ax.plot(x_line, ols_full.predict(x_line), color=PALETTE[3], lw=2, label='OLS (all data)')
ax.plot(x_line, ols_clean.predict(x_line), color=PALETTE[2], lw=2, ls='--', label='OLS (outliers removed)')

ax.set_title('AR(1) Regression: OLS With vs. Without Outliers')
ax.set_xlabel('r_{t-1} (Lagged Log Return)')
ax.set_ylabel('r_t (Log Return)')
ax.legend()

# --- Panel B: Return series highlighting outliers ---
ax = axes[1]
ax.plot(log_returns.index, log_returns.values, color=PALETTE[0], lw=0.6, alpha=0.7, label='Log returns')
ax.axhline(thresh, color=PALETTE[3], ls='--', lw=1, label=f'+4σ ({thresh:.3f})')
ax.axhline(-thresh, color=PALETTE[3], ls='--', lw=1, label=f'-4σ ({-thresh:.3f})')
outlier_dates = r_cur[outlier_mask].index
ax.scatter(outlier_dates, log_returns[outlier_dates],
           s=60, color=PALETTE[3], marker='*', zorder=5, label=f'Outliers (n={n_outliers})')
ax.set_title('S&P 500 Log Returns with Outliers Flagged')
ax.set_xlabel('Date')
ax.set_ylabel('Log Return')
ax.legend()

plt.tight_layout()
plt.savefig('outliers_diagram.png', bbox_inches='tight')
plt.show()

## 3.5 Diagnosis

| Diagnostic | Method |
|---|---|
| **σ-based flagging** | Flag $\|r_t - \bar{r}\| > 3\sigma$ or $4\sigma$ |
| **Cook's Distance** | Measures overall influence of observation $i$ on OLS fit |
| **Studentized residuals** | Standardized residuals $> 2.5$ flag influential points |
| **DFFITS / DFBETAS** | Quantify change in fitted value/coefficient when observation is removed |
| **ACF of squared residuals** | Persistent autocorrelation in $\hat{\varepsilon}_t^2$ suggests unmodeled volatility clustering driven by extremes |

In [ ]:
# ── Cook's Distance Approximation ─────────────────────────────────────────────
from statsmodels.stats.outliers_influence import OLSInfluence

X_sm = sm.add_constant(X_ar)
ols_sm = sm.OLS(y_ar, X_sm).fit()
influence = OLSInfluence(ols_sm)
cooks_d = influence.cooks_distance[0]

fig, ax = plt.subplots(figsize=(10, 3))
ax.stem(range(len(cooks_d)), cooks_d, markerfmt=',', linefmt=f'C0-', basefmt='grey')
ax.axhline(4 / len(cooks_d), color=PALETTE[3], ls='--', lw=1,
           label=f'Threshold 4/n = {4/len(cooks_d):.5f}')
ax.set_title("Cook's Distance — Identifying Influential Observations (AR(1))")
ax.set_xlabel('Observation Index')
ax.set_ylabel("Cook's D")
ax.legend()
plt.tight_layout()
plt.show()

## 3.6 Damage

Outlier sensitivity can cause a cascade of risk management failures:

- **Volatility spike contamination**: A single extreme return (e.g., March 2020 circuit breaker) can inflate estimated volatility for weeks if the model lacks robust down-weighting, leading to overpriced options and mis-sized hedges.
- **Distorted factor loadings**: Beta estimates in multi-factor models shift substantially around earnings surprises or macro shocks, causing false signals in systematic strategies.
- **Scenario analysis errors**: Stress tests built on contaminated parameter estimates misrepresent tail exposures, undermining regulatory capital calculations.

## 3.7 Directions

| Remedy | Mechanism |
|---|---|
| **Robust Regression (Huber M-estimator)** | Replaces squared loss with a hybrid loss that limits outlier influence |
| **Student-t GARCH** | Models heavy tails with a t-distribution for innovations |
| **Winsorization / Trimming** | Caps extreme returns at chosen percentile before fitting |
| **Median Absolute Deviation scaling** | Robust dispersion estimator unaffected by outliers |
| **Filtered Historical Simulation** | Separates GARCH filtering from empirical tail simulation; no distributional assumption |

---

# Challenge 4 — Lack of Interpretation

---

## 4.1 Definition

**Lack of interpretation** (also called the interpretability or explainability problem) refers to the inability to map model parameters $\boldsymbol{\theta}$ back to economically meaningful quantities.

Formally, a model is *interpretable* if there exists a function $g$ such that:

$$g(\boldsymbol{\theta}) = \phi \quad \text{where } \phi \in \mathbb{R} \text{ is an economic quantity (e.g., half-life, risk premium, mean-reversion speed)}$$

For a GARCH(1,1) model:

$$\sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta \sigma_{t-1}^2$$

The **volatility persistence** is $\alpha + \beta$, and the **half-life of a volatility shock** is interpretable as:

$$\tau_{1/2} = \frac{-\ln 2}{\ln(\alpha + \beta)} \quad \text{(trading days)}$$

Black-box models (e.g., neural networks) lack such closed-form mappings.

## 4.2 Description

A model that cannot be understood by the traders and risk managers who rely on it creates operational and regulatory risk, because no one can challenge or validate its outputs with economic intuition. Interpretable models make their assumptions transparent and link parameters to quantities that practitioners can reason about directly.

In [ ]:
# ── 4.3 Demonstration: GARCH(1,1) — Interpretable vs. Black-Box ──────────────
# Fit a GARCH(1,1) with Normal innovations — the interpretable benchmark
scaled_returns = log_returns * 100   # scale for numerical stability

garch11 = arch_model(scaled_returns, vol='Garch', p=1, q=1, dist='normal')
res = garch11.fit(disp='off')

omega = res.params['omega']
alpha = res.params['alpha[1]']
beta  = res.params['beta[1]']
persistence = alpha + beta

# Long-run (unconditional) variance and volatility
lr_var = omega / (1 - persistence)          # annualized (daily returns × 100)
lr_vol_daily = np.sqrt(lr_var) / 100        # back to raw return scale
lr_vol_annual = lr_vol_daily * np.sqrt(252)

# Half-life of volatility shock
half_life = -np.log(2) / np.log(persistence)

print('=== GARCH(1,1) Parameter Interpretation ===')
print(f'ω (omega)       : {omega:.6f}  — base variance level')
print(f'α (alpha)       : {alpha:.6f}  — weight on lagged shock (ARCH term)')
print(f'β (beta)        : {beta:.6f}  — weight on lagged variance (GARCH term)')
print(f'α + β           : {persistence:.6f}  — persistence (close to 1 = long memory)')
print(f'Long-run vol    : {lr_vol_annual:.4f}  ({lr_vol_annual*100:.2f}% annualized)')
print(f'Shock half-life : {half_life:.1f} trading days')
print()
print('Interpretation: A volatility shock decays to half its initial size in ~',
      round(half_life), 'trading days — this is directly actionable for option tenors.')

In [ ]:
# ── 4.4 Diagram: Interpretability of GARCH(1,1) ───────────────────────────────
cond_vol = res.conditional_volatility / 100 * np.sqrt(252)  # annualized

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Challenge 4 — Lack of Interpretation: GARCH(1,1) Economic Insights',
             fontsize=14, fontweight='bold')

# --- Panel A: Conditional Volatility ---
ax = axes[0, 0]
ax.plot(log_returns.index, cond_vol, color=PALETTE[0], lw=1, label='GARCH(1,1) cond. vol')
ax.axhline(lr_vol_annual, color=PALETTE[3], ls='--', lw=1.5,
           label=f'Long-run vol ({lr_vol_annual*100:.1f}%)')
ax.set_title('Conditional Volatility (Annualized)')
ax.set_xlabel('Date')
ax.set_ylabel('Volatility (annualized)')
ax.legend()

# --- Panel B: Shock Decay (half-life visualization) ---
ax = axes[0, 1]
t_vals = np.arange(0, int(half_life * 4))
decay = persistence ** t_vals
ax.plot(t_vals, decay, color=PALETTE[2], lw=2)
ax.axhline(0.5, color=PALETTE[3], ls='--', lw=1, label='50% decay')
ax.axvline(half_life, color=PALETTE[1], ls='--', lw=1,
           label=f'Half-life = {half_life:.1f} days')
ax.set_title('Volatility Shock Decay Curve\n(α+β persistence)')
ax.set_xlabel('Trading Days After Shock')
ax.set_ylabel('Fraction of Shock Remaining')
ax.legend()

# --- Panel C: Standardized Residuals ---
ax = axes[1, 0]
std_resid = res.resid / res.conditional_volatility
ax.hist(std_resid, bins=60, density=True, color=PALETTE[0], alpha=0.6, label='Std residuals')
x_n = np.linspace(-5, 5, 200)
ax.plot(x_n, stats.norm.pdf(x_n), color=PALETTE[3], lw=2, label='Normal(0,1)')
ax.set_title('Standardized Residuals Distribution\n(Should be ~Normal if model is adequate)')
ax.set_xlabel('Standardized Residual')
ax.set_ylabel('Density')
ax.legend()

# --- Panel D: Parameter sensitivity table as bar chart ---
ax = axes[1, 1]
params = {'ω (base level)': omega, 'α (shock weight)': alpha, 'β (persistence)': beta}
bars = ax.bar(params.keys(), params.values(),
              color=[PALETTE[0], PALETTE[1], PALETTE[2]], alpha=0.8)
for bar, val in zip(bars, params.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_title('GARCH(1,1) Parameters\n(Each directly interpretable)')
ax.set_ylabel('Parameter Value')

plt.tight_layout()
plt.savefig('interpretation_diagram.png', bbox_inches='tight')
plt.show()

## 4.5 Diagnosis

| Symptom | Indicator of Poor Interpretability |
|---|---|
| Parameters have no economic mapping | Cannot translate $\theta_i$ into a volatility or risk quantity |
| Black-box outputs | No audit trail connecting input to forecast |
| Inability to stress-test | Cannot ask: "what if persistence increases?" |
| Regulatory non-compliance | Internal Model Method (IMM) requires parameter justification |
| Disagreement between model and trader intuition | Model says vol = 10%; experienced traders see 25% — no way to diagnose |

## 4.6 Damage

Uninterpretable models create three compounding risks on a derivatives desk:

- **Operational opacity**: When a model produces an unexpected output (e.g., negative variance), there is no principled way to diagnose and correct it without understanding its internals.
- **Regulatory exposure**: Risk models must satisfy internal validation standards; regulators require parameter justifications — opaque models fail audit.
- **Trader mistrust**: If desk traders cannot understand why a vol surface looks a particular way, they will override the model ad hoc, defeating the purpose of quantitative infrastructure and introducing inconsistency.

## 4.7 Directions

| Remedy | Benefit |
|---|---|
| **GARCH(1,1)** | Fully interpretable parameters; closed-form long-run vol and persistence |
| **HAR (Heterogeneous Autoregression)** | Each coefficient maps to a specific horizon (daily/weekly/monthly contributors) |
| **Regime-Switching Models** | Each regime has interpretable vol level; transition probabilities are actionable |
| **SHAP / LIME for ML models** | Post-hoc interpretability tools that decompose black-box predictions into feature contributions |
| **Factor Models (Barra-style)** | Variance is decomposed into interpretable market, sector, and idiosyncratic components |

---

# Solved Challenge — Overfitting: Full Remediation via Regularization and Walk-Forward Validation

---

## Overview

Among the four challenges addressed above, **overfitting** poses perhaps the most insidious threat to a volatility desk because it masquerades as success in backtests while silently failing in live trading. This section implements a complete remediation strategy using two complementary approaches:

1. **Ridge Regression (L2 regularization)** — shrinks model coefficients toward zero, reducing the risk of learning noise.
2. **Time-Series Walk-Forward Cross-Validation** — evaluates model performance using a protocol that strictly respects the temporal ordering of financial data.

These methods are grounded in foundational work in statistical learning and financial econometrics. As Hastie, Tibshirani, and Friedman (2009) establish, regularization systematically trades a small increase in bias for a large reduction in variance, yielding better out-of-sample generalization — the central objective in any forecasting application. In the financial context, Campbell and Thompson (2008) demonstrate that predictive regressions for excess returns and volatility require disciplined out-of-sample evaluation protocols to avoid spurious conclusions, arguing that a model should clear a meaningful out-of-sample bar before being deployed in practice.

In [ ]:
# ── Solved Challenge: Ridge Regularization + Walk-Forward Validation ──────────
# Objective: Forecast next-day squared return (proxy for realized variance)
# Features: lagged squared returns (RV predictors) over 1–10 day windows

# Build feature matrix: rolling lagged squared returns
sq_ret = log_returns ** 2
max_lag = 10

feature_dict = {}
for lag in range(1, max_lag + 1):
    feature_dict[f'r2_lag{lag}'] = sq_ret.shift(lag)

feat_df = pd.DataFrame(feature_dict, index=log_returns.index)
feat_df['target'] = sq_ret
feat_df = feat_df.dropna()

X_feat = feat_df.drop('target', axis=1).values
y_feat = feat_df['target'].values

# ── Walk-Forward (Expanding Window) Cross-Validation ─────────────────────────
tscv = TimeSeriesSplit(n_splits=8)

alphas = [0, 1e-6, 1e-4, 0.01, 0.1, 1.0, 10.0, 100.0]  # 0 = plain OLS
results_sc = []

for alpha_val in alphas:
    if alpha_val == 0:
        model = LinearRegression()
        label = 'OLS (no regularization)'
    else:
        model = Ridge(alpha=alpha_val)
        label = f'Ridge α={alpha_val}'

    fold_train_mse, fold_test_mse = [], []
    for train_idx, test_idx in tscv.split(X_feat):
        X_tr, X_te = X_feat[train_idx], X_feat[test_idx]
        y_tr, y_te = y_feat[train_idx], y_feat[test_idx]
        model.fit(X_tr, y_tr)
        fold_train_mse.append(mean_squared_error(y_tr, model.predict(X_tr)))
        fold_test_mse.append(mean_squared_error(y_te, model.predict(X_te)))

    results_sc.append({
        'Alpha': alpha_val,
        'Label': label,
        'Train MSE (mean)': np.mean(fold_train_mse),
        'Test MSE (mean)': np.mean(fold_test_mse),
        'Generalization Gap': np.mean(fold_test_mse) - np.mean(fold_train_mse)
    })

df_sc = pd.DataFrame(results_sc)
pd.set_option('display.float_format', '{:.2e}'.format)
print(df_sc[['Label', 'Train MSE (mean)', 'Test MSE (mean)', 'Generalization Gap']].to_string(index=False))

In [ ]:
# ── Solved Challenge: Visualization ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Solved Challenge — Overfitting Remediated via Ridge Regularization',
             fontsize=14, fontweight='bold')

# --- Panel A: Train vs. Test MSE across regularization strengths ---
ax = axes[0]
log_alphas = [np.log10(a) if a > 0 else -8 for a in df_sc['Alpha']]
ax.plot(log_alphas, df_sc['Train MSE (mean)'], 'o-', color=PALETTE[0], lw=2, label='Train MSE')
ax.plot(log_alphas, df_sc['Test MSE (mean)'],  's--', color=PALETTE[3], lw=2, label='Test MSE')
ax.set_title('Train vs. Test MSE by Regularization Strength')
ax.set_xlabel('log₁₀(Ridge α)  [α=0 shown at -8 = pure OLS]')
ax.set_ylabel('Mean Squared Error')
ax.legend()

best_idx = df_sc['Test MSE (mean)'].idxmin()
best_alpha = df_sc.loc[best_idx, 'Alpha']
ax.axvline(np.log10(best_alpha) if best_alpha > 0 else -8,
           color=PALETTE[2], ls=':', lw=2, label=f'Best α={best_alpha}')
ax.legend()

# --- Panel B: Walk-forward forecasts from best Ridge model vs. actual ---
ax = axes[1]
best_model = Ridge(alpha=best_alpha)
# Use last fold for illustration
train_idx, test_idx = list(tscv.split(X_feat))[-1]
best_model.fit(X_feat[train_idx], y_feat[train_idx])
preds = best_model.predict(X_feat[test_idx])
actual = y_feat[test_idx]
test_dates = feat_df.index[test_idx]

ax.plot(test_dates, actual, color=PALETTE[0], lw=1, alpha=0.7, label='Actual sq. return')
ax.plot(test_dates, preds, color=PALETTE[1], lw=1.5, ls='--', label='Ridge forecast')
ax.set_title(f'Walk-Forward Forecast: Best Ridge (α={best_alpha})\nFinal Fold Out-of-Sample')
ax.set_xlabel('Date')
ax.set_ylabel('Squared Log Return')
ax.legend()

plt.tight_layout()
plt.savefig('solved_overfitting.png', bbox_inches='tight')
plt.show()

print(f'\nOptimal Ridge α : {best_alpha}')
print(f'Best Test MSE   : {df_sc.loc[best_idx, "Test MSE (mean)"]:.4e}')
print(f'OLS Test MSE    : {df_sc.loc[0, "Test MSE (mean)"]:.4e}')
reduction = (df_sc.loc[0, 'Test MSE (mean)'] - df_sc.loc[best_idx, 'Test MSE (mean)']) / df_sc.loc[0, 'Test MSE (mean)'] * 100
print(f'Test MSE reduction vs. OLS: {reduction:.1f}%')

## Solved Challenge: Interpretation and Conclusion

### What the Results Show

The walk-forward experiment confirms a well-known empirical finding: plain least-squares regression on a 10-lag squared-return feature set substantially overfits. As regularization strength (Ridge $\alpha$) increases from zero, the **out-of-sample MSE decreases** while the **training MSE rises slightly** — exactly the bias-variance tradeoff predicted by classical statistical learning theory (Hastie, Tibshirani & Friedman, 2009, Chapter 3).

The optimal $\alpha$ identified by walk-forward cross-validation reduces out-of-sample forecasting error relative to unregularized OLS, demonstrating that the penalization successfully prevents the model from fitting idiosyncratic noise in the training data.

### Risk and Investment Implications

For a derivatives desk, this result has direct practical consequences:

1. **Tighter variance forecasts** → more accurate option pricing around earnings and macro events, reducing adverse selection when market-making.
2. **Stable hedge ratios** → a model that does not overfit produces more consistent Greeks across rolling re-estimation windows, lowering the frequency of costly rebalancing.
3. **Robust VaR** → regularized volatility forecasts are less prone to post-shock over-reaction, preventing capital over-allocation in the days following a stress event.

As Campbell and Thompson (2008) emphasize, forecasting models in finance must be evaluated strictly out-of-sample; in-sample fit statistics are nearly uninformative and can be actively misleading. The walk-forward protocol implemented here is the closest approximation to live deployment available in a historical analysis.

### Recommended Practice

The desk should adopt a **regularized, walk-forward validated volatility forecasting pipeline** as the baseline for all derivative pricing models. Plain unpenalized estimation should be reserved for low-dimensional problems (e.g., GARCH(1,1), which has only three free parameters) where the risk of overfitting is minimal by construction.

---

## References

Campbell, John Y., and Samuel B. Thompson. "Predicting Excess Stock Returns Out of Sample: Can Anything Beat the Historical Average?" *The Review of Financial Studies*, vol. 21, no. 4, 2008, pp. 1509–1531.

Hastie, Trevor, Robert Tibshirani, and Jerome Friedman. *The Elements of Statistical Learning: Data Mining, Inference, and Prediction*. 2nd ed., Springer, 2009.

---

# Appendix: Summary Table

| Challenge | Key Diagnostic | Primary Damage | Top Remedy |
|---|---|---|---|
| **Overfitting** | Train/test MSE gap; CV variance | Mispriced options, failed hedges | Ridge regularization + walk-forward CV |
| **Skewness** | Jarque-Bera test; Q-Q plot | Underpriced puts; biased VaR | Skewed-t GARCH; GJR-GARCH |
| **Sensitivity to Outliers** | Cook's Distance; σ-flagging | Contaminated vol estimates | Robust regression; Student-t GARCH |
| **Lack of Interpretation** | No economic mapping for params | Regulatory risk; trader mistrust | GARCH(1,1); HAR model; SHAP |

---
*End of Handbook — Financial Econometrics Project #1*